In [ ]:
import os, time, zlib, tempfile, random
from collections import Counter
import numpy as np, pandas as pd
from gensim.models import Word2Vec
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
from IPython.display import display
from cooccurrence import (tokenize, build_vocabulary, build_cooccurrence_matrix, cosine_similarity,
                          most_similar, matrix_stats, cbow_pairs, skipgram_pairs, ppmi, svd_embeddings,
                          generate_corpus)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RESULTS = [] 
def log(experiment, setting, item, metric, value):
    RESULTS.append(dict(experiment=experiment, setting=setting, item=item, metric=metric,
                        value=round(float(value), 4) if isinstance(value, (int, float, np.floating)) else value))
SEED = 42
corpus = generate_corpus(30000, seed=SEED)
sentences = [s for s, _ in corpus]; topics = [t for _, t in corpus]
tokens = [s.split() for s in sentences]
token_sets = [set(t) for t in tokens]
cnt = Counter(w for t in tokens for w in t)
print(f"câu: {len(sentences):,} | token: {sum(cnt.values()):,} | vocabulary: {len(cnt)} từ")
print("ví dụ:", sentences[:3])
print("tần suất:", {w: cnt[w] for w in ['doctor','physician','nurse','hospital','patient','banana','car','bank']})

câu: 30,000 | token: 207,997 | vocabulary: 212 từ
ví dụ: ['the doctor prescribed insulin for the cancer', 'france is a beautiful country', 'clinical surgery of disease requires a doctor']
tần suất: {'doctor': 1769, 'physician': 1725, 'nurse': 1366, 'hospital': 2558, 'patient': 3209, 'banana': 507, 'car': 742, 'bank': 1746}


## Exp1

In [2]:
STOP = ["the","a","an","of","in","on","at","for","to","and","is","was","with","i","my"]
small = sentences[:2000]
rows = []
for k in (1, 2, 5):
    v = build_vocabulary(small, min_count=1, stopwords=STOP)
    X = build_cooccurrence_matrix(small, v, window=k)
    st = matrix_stats(X)
    c = lambda a, b: cosine_similarity(X[v[a]], X[v[b]])
    rows.append({"window": k, "vocab_size": st["vocab_size"], "matrix": st["matrix_shape"],
                 "entries": st["n_entries"], "non-zero": st["nonzero"], "density %": round(100*st["density"], 2),
                 "cos(doctor,physician)": round(c("doctor","physician"), 3),
                 "cos(doctor,hospital)": round(c("doctor","hospital"), 3),
                 "cos(doctor,banana)": round(c("doctor","banana"), 3)})
    for m in ("vocab_size","n_entries","nonzero","density"):
        log("exp1_word_context", f"window={k}", "matrix", m, st[m])
    for a, b in [("doctor","physician"),("doctor","hospital"),("doctor","banana")]:
        log("exp1_word_context", f"window={k}", f"{a}-{b}", "cosine_count", c(a, b))
exp1 = pd.DataFrame(rows); display(exp1)

,window,vocab_size,matrix,entries,non-zero,density %,"cos(doctor,physician)","cos(doctor,hospital)","cos(doctor,banana)"
0,1,197,197x197,38809,486,1.25,0.645,0.000,0.0
1,2,197,197x197,38809,1038,2.67,0.665,0.000,0.0
2,5,197,197x197,38809,2196,5.66,0.713,0.613,0.0


In [ ]:
V_all = build_vocabulary(sentences, min_count=1)
X_all = build_cooccurrence_matrix(sentences, V_all, window=5)
V_ns  = build_vocabulary(sentences, min_count=1, stopwords=STOP)
X_ns  = build_cooccurrence_matrix(sentences, V_ns, window=5)
P_ns  = ppmi(X_ns)
E_svd = svd_embeddings(X_ns, dim=20)
def show(name, M, V):
    top = most_similar("doctor", M, V, top_k=5)
    print(f"{name:34s}", [(w, round(s, 3)) for w, s in top])
show("count, giữ 'the' & stopword", X_all, V_all)
show("count, bỏ stopword", X_ns, V_ns)
show("PPMI", P_ns, V_ns)
show("PPMI + SVD (dense, 20 chiều)", E_svd, V_ns)
for name, M, V in [("count_with_stopwords", X_all, V_all), ("count_no_stopwords", X_ns, V_ns),
                   ("ppmi", P_ns, V_ns), ("ppmi_svd20", E_svd, V_ns)]:
    for rank, (w, s) in enumerate(most_similar("doctor", M, V, top_k=5), 1):
        log("exp1_doctor_top5", name, f"doctor->{w}", f"cosine_rank{rank}", s)
rows = []
for k in (1, 2, 3, 5):
    Xk = build_cooccurrence_matrix(sentences, V_ns, window=k)
    ch, cp = cosine_similarity(Xk[V_ns["doctor"]], Xk[V_ns["hospital"]]), cosine_similarity(Xk[V_ns["doctor"]], Xk[V_ns["physician"]])
    rows.append({"window": k, "nonzero": int(np.count_nonzero(Xk)), "cos(doctor,hospital)": round(ch, 3), "cos(doctor,physician)": round(cp, 3)})
    log("exp1_full_corpus_count", f"window={k}", "doctor-hospital", "cosine_count", ch); log("exp1_full_corpus_count", f"window={k}", "doctor-physician", "cosine_count", cp)
display(pd.DataFrame(rows))

count, giữ 'the' & stopword        [('nurse', 0.998), ('surgeon', 0.975), ('physician', 0.962), ('patient', 0.916), ('the', 0.91)]
count, bỏ stopword                 [('nurse', 0.991), ('surgeon', 0.855), ('physician', 0.769), ('hospital', 0.651), ('admitted', 0.59)]
PPMI                               [('nurse', 0.995), ('surgeon', 0.71), ('physician', 0.622), ('patient', 0.5), ('hospital', 0.471)]
PPMI + SVD (dense, 20 chiều)       [('nurse', 1.0), ('surgeon', 0.894), ('hospital', 0.807), ('patient', 0.805), ('physician', 0.616)]


,window,nonzero,"cos(doctor,hospital)","cos(doctor,physician)"
0,1,1094,0.000,0.683
1,2,2775,0.000,0.741
2,3,4763,0.495,0.782
3,5,7390,0.651,0.769


## 3. Core implementation — kiểm tra bằng ví dụ của đề

In [4]:
corpus1 = ["the cat eats fish", "the dog eats fish", "the cat likes milk", "the dog likes meat"]
vocab1 = build_vocabulary(corpus1, fixed_words=["cat","dog","eats","likes","fish","milk","meat"])
X1 = build_cooccurrence_matrix(corpus1, vocab1, window=1)
expected = {"cat": [0,0,1,1,0,0,0], "dog": [0,0,1,1,0,0,0], "eats": [1,1,0,0,2,0,0], "likes": [1,1,0,0,0,1,1]}
for w, e in expected.items():
    assert X1[vocab1[w]].astype(int).tolist() == e, w
assert abs(cosine_similarity(X1[0], X1[1]) - 1.0) < 1e-12
print(pd.DataFrame(X1.astype(int), index=vocab1, columns=vocab1).loc[["cat","dog","eats","likes"]])
print("most_similar('cat'):", most_similar("cat", X1, vocab1, top_k=3))
print("OK: khớp kết quả tính tay (calculations.md, Bài 1)")

       cat  dog  eats  likes  fish  milk  meat
cat      0    0     1      1     0     0     0
dog      0    0     1      1     0     0     0
eats     1    1     0      0     2     0     0
likes    1    1     0      0     0     1     1
most_similar('cat'): [('dog', 0.9999999999999998), ('fish', 0.7071067811865475), ('meat', 0.7071067811865475)]
OK: khớp kết quả tính tay (calculations.md, Bài 1)


## 4. CBOW vs Skip-gram

In [5]:
sent = "the cat eats fish".split()
cb, sg = cbow_pairs(sent, 1), skipgram_pairs(sent, 1)
print("CBOW (context -> target):");  [print("  ", c, "->", t) for c, t in cb]
print("Skip-gram (target -> context):"); [print("  ", t, "->", c) for t, c in sg]
assert cb == [(["cat"],"the"), (["the","eats"],"cat"), (["cat","fish"],"eats"), (["eats"],"fish")]
assert sg == [("the","cat"),("cat","the"),("cat","eats"),("eats","cat"),("eats","fish"),("fish","eats")]
print(f"CBOW: {len(cb)} mẫu | Skip-gram: {len(sg)} cặp")

CBOW (context -> target):
   ['cat'] -> the
   ['the', 'eats'] -> cat
   ['cat', 'fish'] -> eats
   ['eats'] -> fish
Skip-gram (target -> context):
   the -> cat
   cat -> the
   cat -> eats
   eats -> cat
   eats -> fish
   fish -> eats
CBOW: 4 mẫu | Skip-gram: 6 cặp


In [ ]:
def HASH(s):
    return zlib.crc32(s.encode("utf8"))
def train(toks, vector_size=100, window=5, min_count=2, epochs=10, sg=1, negative=5, seed=SEED):
    t0 = time.time()
    m = Word2Vec(sentences=toks, vector_size=vector_size, window=window, min_count=min_count,
                 epochs=epochs, sg=sg, negative=negative, hs=0, workers=1, seed=seed, hashfxn=HASH)
    return m, time.time() - t0
model, t_train = train(tokens)
wv = model.wv
print(f"vocabulary={len(wv)} | vector_size={wv.vector_size} | corpus_count={model.corpus_count:,} | "
      f"epochs={model.epochs} | thời gian train={t_train:.1f}s")
for a, b in [("doctor","physician"),("doctor","hospital"),("cat","dog"),("doctor","banana"),("car","banana")]:
    print(f"cos({a},{b}) = {wv.similarity(a,b):.3f}")
    log("exp2_word2vec_baseline", "sg d=100 w=5", f"{a}-{b}", "cosine", wv.similarity(a, b))
log("exp2_word2vec_baseline", "sg d=100 w=5", "model", "vocab_size", len(wv))
log("exp2_word2vec_baseline", "sg d=100 w=5", "model", "train_seconds", t_train)

m_cbow, t_cbow = train(tokens, sg=0)
print(f"\n[CBOW  sg=0] cos(doctor,physician)={m_cbow.wv.similarity('doctor','physician'):.3f}, "
      f"top-5 doctor: {[w for w,_ in m_cbow.wv.most_similar('doctor', topn=5)]}  ({t_cbow:.1f}s)")
print(f"[Skip-gram ] cos(doctor,physician)={wv.similarity('doctor','physician'):.3f}, "
      f"top-5 doctor: {[w for w,_ in wv.most_similar('doctor', topn=5)]}")
log("exp2_cbow_vs_skipgram", "cbow", "doctor-physician", "cosine", m_cbow.wv.similarity("doctor","physician"))
log("exp2_cbow_vs_skipgram", "skipgram", "doctor-physician", "cosine", wv.similarity("doctor","physician"))
five = ["doctor","physician","hospital","banana","car"]
M5 = pd.DataFrame([[wv.similarity(a, b) for b in five] for a in five], index=five, columns=five).round(3); display(M5)
pairs5 = sorted(((wv.similarity(a, b), a, b) for i, a in enumerate(five) for b in five[i+1:]), reverse=True)
print("Xếp hạng cặp:", [f"{a}–{b}: {s:.3f}" for s, a, b in pairs5])
for s_, a, b in pairs5: log("pred1_five_words", "sg d=100 w=5", f"{a}-{b}", "cosine", s_)
Wn = wv.vectors / np.linalg.norm(wv.vectors, axis=1, keepdims=True); G = Wn @ Wn.T
base = (G.sum() - len(G)) / (len(G) * (len(G) - 1))
print(f"Cosine trung bình giữa MỌI cặp từ khác nhau trong vocabulary = {base:.3f}  (mức nền: cosine tuyệt đối không được hiệu chỉnh quanh 0)")
log("pred1_five_words", "sg d=100 w=5", "all word pairs", "mean_cosine_baseline", base)

vocabulary=212 | vector_size=100 | corpus_count=30,000 | epochs=10 | thời gian train=1.8s
cos(doctor,physician) = 0.737
cos(doctor,hospital) = 0.389
cos(cat,dog) = 0.810
cos(doctor,banana) = 0.237
cos(car,banana) = 0.461



[CBOW  sg=0] cos(doctor,physician)=0.834, top-5 doctor: ['nurse', 'surgeon', 'physician', 'girl', 'woman']  (0.9s)
[Skip-gram ] cos(doctor,physician)=0.737, top-5 doctor: ['nurse', 'surgeon', 'physician', 'rest', 'should']


,doctor,physician,hospital,banana,car
doctor,1.000,0.737,0.389,0.237,0.253
physician,0.737,1.000,0.321,0.110,0.192
hospital,0.389,0.321,1.000,0.220,0.177
banana,0.237,0.110,0.220,1.000,0.461
car,0.253,0.192,0.177,0.461,1.000


Xếp hạng cặp: ['doctor–physician: 0.737', 'banana–car: 0.461', 'doctor–hospital: 0.389', 'physician–hospital: 0.321', 'doctor–car: 0.253', 'doctor–banana: 0.237', 'hospital–banana: 0.220', 'physician–car: 0.192', 'hospital–car: 0.177', 'physician–banana: 0.110']
Cosine trung bình giữa MỌI cặp từ khác nhau trong vocabulary = 0.287  (mức nền: cosine tuyệt đối không được hiệu chỉnh quanh 0)


In [7]:
inv_ns = {i: w for w, i in V_ns.items()}
def evidence(w, n, k=5):
    both = sum(1 for t in token_sets if w in t and n in t)
    if w not in V_ns or n not in V_ns: return both, []
    shared = sorted(((min(X_ns[V_ns[w], j], X_ns[V_ns[n], j]), inv_ns[j]) for j in range(len(V_ns))), reverse=True)[:k]
    return both, [f"{c}({int(v)})" for v, c in shared if v > 0]

inspect_words = ["doctor","hospital","patient","disease","computer","football","banana"]
rows = []
for w in inspect_words:
    for rank, (n, s) in enumerate(wv.most_similar(w, topn=5), 1):
        both, shared = evidence(w, n)
        rows.append({"word": w, "rank": rank, "neighbor": n, "cosine": round(s, 3),
                     "#câu chứa cả hai": both, "context chung (top-5)": ", ".join(shared)})
        log("exp2_top5", "sg d=100 w=5", f"{w}->{n}", f"cosine_rank{rank}", s)
insp = pd.DataFrame(rows)
for w in inspect_words:
    print(f"\n### {w}"); print(insp[insp.word == w].drop(columns="word").to_string(index=False))


### doctor
 rank  neighbor  cosine  #câu chứa cả hai                                                 context chung (top-5)
    1     nurse   0.989                 0 patient(499), symptoms(142), explained(142), works(137), treated(136)
    2   surgeon   0.791                 1      patient(299), surgery(111), about(111), asked(109), treated(104)
    3 physician   0.737                 0 patient(503), recommended(161), provided(161), examined(161), by(161)
    4      rest   0.572               177                                     said(183), should(177), doctor(2)
    5    should   0.555               177                                       said(183), rest(177), doctor(2)

### hospital
 rank neighbor  cosine  #câu chứa cả hai                                                  context chung (top-5)
    1     with   0.514               595                                                                       
    2     long   0.492               283          works(284), shifts(283), doc

## Exp3

In [8]:
PAIRS = [("doctor","physician"), ("doctor","hospital"), ("cat","dog"), ("doctor","banana")]
rows, models_w = [], {}
for win in (2, 5, 10):
    vals = {p: [] for p in PAIRS}; times = []
    for sd in (42, 43, 44):
        m, t = train(tokens, window=win, seed=sd); times.append(t)
        if sd == 42: models_w[win] = m
        for a, b in PAIRS: vals[(a, b)].append(m.wv.similarity(a, b))
    row = {"window": win, "train_s": round(np.mean(times), 1)}
    for (a, b), v in vals.items():
        row[f"{a}–{b}"] = f"{np.mean(v):.3f} ± {np.std(v):.3f}"
        log("exp3_window", f"window={win}", f"{a}-{b}", "cosine_mean_3seeds", np.mean(v))
        log("exp3_window", f"window={win}", f"{a}-{b}", "cosine_std_3seeds", np.std(v))
    rows.append(row)
display(pd.DataFrame(rows))
for win, m in models_w.items():
    print(f"window={win:2d} top-5 doctor:", [(w, round(s, 2)) for w, s in m.wv.most_similar("doctor", topn=5)])

,window,train_s,doctor–physician,doctor–hospital,cat–dog,doctor–banana
0,2,1.6,0.749 ± 0.007,0.375 ± 0.009,0.845 ± 0.010,0.256 ± 0.027
1,5,2.1,0.739 ± 0.003,0.384 ± 0.004,0.797 ± 0.009,0.221 ± 0.015
2,10,2.0,0.732 ± 0.007,0.361 ± 0.004,0.780 ± 0.006,0.218 ± 0.007


window= 2 top-5 doctor: [('nurse', 0.99), ('surgeon', 0.79), ('physician', 0.75), ('home', 0.58), ('shifts', 0.51)]
window= 5 top-5 doctor: [('nurse', 0.99), ('surgeon', 0.79), ('physician', 0.74), ('rest', 0.57), ('should', 0.55)]
window=10 top-5 doctor: [('nurse', 0.99), ('surgeon', 0.79), ('physician', 0.73), ('rest', 0.58), ('should', 0.56)]


## Exp4

In [9]:
GOLD = [("doctor","physician",9.5),("car","automobile",9.5),("laptop","computer",8.5),("king","queen",8.0),
        ("cat","dog",7.5),("car","truck",7.5),("banana","apple",7.5),("man","woman",7.5),("doctor","nurse",7.0),
        ("football","goal",7.0),("paris","france",7.0),("doctor","hospital",6.5),("patient","disease",6.0),
        ("cat","fish",5.0),("doctor","computer",1.0),("computer","banana",0.5),("king","banana",0.3),("hospital","football",0.3)]
ANALOGIES = [(["king","woman"],["man"],"queen"), (["prince","woman"],["man"],"princess"),
             (["queen","man"],["woman"],"king"), (["princess","man"],["woman"],"prince"),
             (["boy","woman"],["man"],"girl"),
             (["paris","italy"],["france"],"rome"), (["rome","france"],["italy"],"paris"),
             (["berlin","spain"],["germany"],"madrid"), (["madrid","germany"],["spain"],"berlin"),
             (["tokyo","vietnam"],["japan"],"hanoi"), (["hanoi","japan"],["vietnam"],"tokyo"),
             (["berlin","italy"],["germany"],"rome")]
def spearman_w2v(wv_):
    pred = [wv_.similarity(a, b) for a, b, _ in GOLD]
    return spearmanr(pred, [g for *_, g in GOLD]).statistic
def analogy_acc(wv_):
    t1 = t5 = 0
    for pos, neg, ans in ANALOGIES:
        top = [w for w, _ in wv_.most_similar(positive=pos, negative=neg, topn=5)]
        t1 += top[0] == ans; t5 += ans in top
    return t1 / len(ANALOGIES), t5 / len(ANALOGIES)

keep = [i for i, t in enumerate(topics) if t != "noise"]
y = np.array([topics[i] for i in keep]); idx_tr, idx_te = train_test_split(np.arange(len(keep)), test_size=0.2, random_state=0, stratify=y)
def downstream(wv_):
    Xf = np.vstack([wv_.get_mean_vector(tokens[i]) for i in keep])
    clf = LogisticRegression(max_iter=2000).fit(Xf[idx_tr], y[idx_tr])
    return clf.score(Xf[idx_te], y[idx_te])

rows, models_d = [], {}
for d in (50, 100, 300):
    m, t = train(tokens, vector_size=d); models_d[d] = m
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "m.model"); m.save(path)
        size_kb = sum(os.path.getsize(os.path.join(tmp, f)) for f in os.listdir(tmp)) / 1024
    a1, a5 = analogy_acc(m.wv); rho = spearman_w2v(m.wv); acc = downstream(m.wv)
    rows.append({"dim": d, "train_s": round(t, 1), "model_KB": round(size_kb), "cos(doctor,physician)": round(m.wv.similarity("doctor","physician"), 3),
                 "cos(doctor,banana)": round(m.wv.similarity("doctor","banana"), 3), "Spearman(18 pairs)": round(rho, 3),
                 "analogy top-1": round(a1, 3), "analogy top-5": round(a5, 3), "downstream acc": round(acc, 4)})
    for k, v in [("train_seconds", t), ("model_KB", size_kb), ("spearman_gold", rho), ("analogy_top1", a1), ("analogy_top5", a5), ("downstream_acc", acc),
                 ("cos_doctor_physician", m.wv.similarity("doctor","physician")), ("cos_doctor_banana", m.wv.similarity("doctor","banana"))]:
        log("exp4_dimension", f"dim={d}", "model", k, v)
exp4 = pd.DataFrame(rows); display(exp4)

,dim,train_s,model_KB,"cos(doctor,physician)","cos(doctor,banana)",Spearman(18 pairs),analogy top-1,analogy top-5,downstream acc
0,50,1.9,96,0.743,0.258,0.801,0.750,1.0,1.0
1,100,2.0,179,0.737,0.237,0.796,0.833,1.0,1.0
2,300,3.1,510,0.736,0.260,0.803,0.833,1.0,1.0


## Eval

In [ ]:
def pair_cos(M, V, a, b): return cosine_similarity(M[V[a]], M[V[b]])
reps = {"count (window=5)": lambda a, b: pair_cos(X_ns, V_ns, a, b),
        "PPMI (window=5)": lambda a, b: pair_cos(P_ns, V_ns, a, b),
        "PPMI+SVD dense 20d": lambda a, b: pair_cos(E_svd, V_ns, a, b),
        "Word2Vec sg d=100": lambda a, b: float(wv.similarity(a, b))}
gold = [g for *_, g in GOLD]
summ = []
for name, f in reps.items():
    sc = [f(a, b) for a, b, _ in GOLD]
    rho = spearmanr(sc, gold).statistic
    summ.append({"representation": name, "Spearman ρ vs gold": round(rho, 3)})
    log("eval_similarity", name, "18 pairs", "spearman_gold", rho)
    for (a, b, g), s in zip(GOLD, sc): log("eval_similarity_pairs", name, f"{a}-{b}", "cosine", s)
display(pd.DataFrame(summ))

df = pd.DataFrame([{"pair": f"{a}–{b}", "gold": g, "w2v_cos": round(float(wv.similarity(a, b)), 3)} for a, b, g in GOLD])
df["gold_rank"] = df.gold.rank(ascending=False, method="average"); df["w2v_rank"] = df.w2v_cos.rank(ascending=False, method="average")
df["rank_gap"] = (df.w2v_rank - df.gold_rank).round(1)
display(df.sort_values("w2v_cos", ascending=False).reset_index(drop=True))
print("\nCác trường hợp bất ngờ (|rank_gap| lớn nhất):")
display(df.reindex(df.rank_gap.abs().sort_values(ascending=False).index).head(5))

,representation,Spearman ρ vs gold
0,count (window=5),0.816
1,PPMI (window=5),0.838
2,PPMI+SVD dense 20d,0.686
3,Word2Vec sg d=100,0.796


,pair,gold,w2v_cos,gold_rank,w2v_rank,rank_gap
0,doctor–nurse,7.0,0.989,10.0,1.0,-9.0
1,car–automobile,9.5,0.982,1.5,2.0,0.5
2,car–truck,7.5,0.968,6.5,3.0,-3.5
3,banana–apple,7.5,0.937,6.5,4.0,-2.5
4,laptop–computer,8.5,0.934,3.0,5.0,2.0
5,cat–dog,7.5,0.810,6.5,6.0,-0.5
6,man–woman,7.5,0.780,6.5,7.0,0.5
7,king–queen,8.0,0.776,4.0,8.0,4.0
8,doctor–physician,9.5,0.737,1.5,9.0,7.5
9,football–goal,7.0,0.551,10.0,10.0,0.0



Các trường hợp bất ngờ (|rank_gap| lớn nhất):


,pair,gold,w2v_cos,gold_rank,w2v_rank,rank_gap
8,doctor–nurse,7.0,0.989,10.0,1.0,-9.0
0,doctor–physician,9.5,0.737,1.5,9.0,7.5
3,king–queen,8.0,0.776,4.0,8.0,4.0
5,car–truck,7.5,0.968,6.5,3.0,-3.5
14,doctor–computer,1.0,0.218,15.0,18.0,3.0


## Word Analogy

In [ ]:
for pos, neg in [(["king","woman"], ["man"]), (["paris","italy"], ["france"])]:
    print(f"{' + '.join(pos)} - {' - '.join(neg)} ->", [(w, round(s, 3)) for w, s in wv.most_similar(positive=pos, negative=neg, topn=5)])
off = lambda a, b: wv[a] - wv[b]
for (a, b), (c, d) in [(("king","man"), ("queen","woman")), (("paris","france"), ("rome","italy")), (("king","man"), ("rome","italy"))]:
    v = cosine_similarity(off(a, b), off(c, d)); print(f"cos({a}-{b}, {c}-{d}) = {v:.3f}")
    log("eval_analogy_offsets", "sg d=100 w=5", f"({a}-{b}) vs ({c}-{d})", "cosine", v)
rows = []
for pos, neg, ans in ANALOGIES:
    top = [w for w, _ in wv.most_similar(positive=pos, negative=neg, topn=5)]
    rows.append({"analogy": f"{pos[0]} - {neg[0]} + {pos[1]}", "expected": ans, "top-1": top[0], "hit@1": top[0] == ans, "hit@5": ans in top})
    log("eval_analogy", "sg d=100 w=5", rows[-1]["analogy"], "hit_at_1", int(top[0] == ans))
display(pd.DataFrame(rows))
a1, a5 = analogy_acc(wv); print(f"top-1 = {a1:.2f} | top-5 = {a5:.2f}")
log("eval_analogy", "sg d=100 w=5", "12 analogies", "top1_acc", a1); log("eval_analogy", "sg d=100 w=5", "12 analogies", "top5_acc", a5)
king, man, woman = np.array([8,2,7]), np.array([5,1,5]), np.array([5,3,5])
new = king - man + woman
print("\nking - man + woman =", new, "| cos(new, king) =", round(cosine_similarity(new, king), 4))

king + woman - man -> [('princess', 0.898), ('queen', 0.876), ('prince', 0.78), ('girl', 0.576), ('throne', 0.517)]
paris + italy - france -> [('rome', 0.663), ('madrid', 0.627), ('hanoi', 0.609), ('tokyo', 0.584), ('berlin', 0.547)]
cos(king-man, queen-woman) = 0.858
cos(paris-france, rome-italy) = 0.522
cos(king-man, rome-italy) = 0.083


,analogy,expected,top-1,hit@1,hit@5
0,king - man + woman,queen,princess,False,True
1,prince - man + woman,princess,princess,True,True
2,queen - woman + man,king,king,True,True
3,princess - woman + man,prince,king,False,True
4,boy - man + woman,girl,girl,True,True
5,paris - france + italy,rome,rome,True,True
6,rome - italy + france,paris,paris,True,True
7,berlin - germany + spain,madrid,madrid,True,True
8,madrid - spain + germany,berlin,berlin,True,True
9,tokyo - japan + vietnam,hanoi,hanoi,True,True


top-1 = 0.83 | top-5 = 1.00

king - man + woman = [8 4 7] | cos(new, king) = 0.9849


## 11. Application — Semantic search / clustering

In [12]:
DOCS = {
 "d1": "the physician recommended a new therapy for the patient with diabetes",
 "d2": "the clinic offers clinical care and medicine for every patient",
 "d3": "the hospital hired a surgeon to perform surgery on the cancer patient",
 "d4": "the football team won the match at the stadium",
 "d5": "the car needs fuel and a new engine",
 "d6": "the programmer wrote software code on the laptop",
 "d7": "the bank approved the loan and charged interest",
 "d8": "the banana is a sweet fruit sold at the market",
 "d9": "the king ruled the kingdom from the palace",
 "d10": "the doctor prescribed antibiotics for the infection",
 "d11": "paris is the capital of france",
 "d12": "the nurse visited the patient at home"}
RELEVANT = {"medical treatment": {"d1","d2","d3","d10","d12"}, "automobile engine": {"d5"}, "paris capital": {"d11"}}
ids = list(DOCS); texts = [DOCS[i] for i in ids]
tfidf = TfidfVectorizer(stop_words=STOP + ["the"]).fit(texts); Dm = tfidf.transform(texts)

def embed_text(t):
    ws = [w for w in t.split() if w not in STOP and w in wv]
    return wv.get_mean_vector(ws) if ws else np.zeros(wv.vector_size)
Demb = np.vstack([embed_text(t) for t in texts])
def search(query, k=12):
    qs = tfidf.transform([query]).toarray()[0]
    tf = [(ids[i], float(cosine_similarity(qs, Dm[i].toarray()[0]))) for i in range(len(ids))]
    em = [(ids[i], float(cosine_similarity(embed_text(query), Demb[i]))) for i in range(len(ids))]
    return sorted(tf, key=lambda x: -x[1])[:k], sorted(em, key=lambda x: -x[1])[:k]
def rprec(res, rel): return sum(d in rel for d, s in res[:len(rel)] if s > 0) / len(rel)   # R-precision: R = số doc liên quan

for q, rel in RELEVANT.items():
    qw = [w for w in q.split() if w in wv]
    print(f"\n=== Query: '{q}' ===")
    print("từ liên quan (embedding):", {w: [n for n, _ in wv.most_similar(w, topn=5)] for w in qw})
    tf, em = search(q)
    print("TF-IDF   :", [(d, round(s, 2)) for d, s in tf[:4]], f"| R-precision = {rprec(tf, rel):.2f}  (liên quan: {sorted(rel)})")
    print("Embedding:", [(d, round(s, 2)) for d, s in em[:4]], f"| R-precision = {rprec(em, rel):.2f}")
    log("app_semantic_search", q, "TF-IDF", "r_precision", rprec(tf, rel)); log("app_semantic_search", q, "embedding_mean", "r_precision", rprec(em, rel))


=== Query: 'medical treatment' ===
từ liên quan (embedding): {'medical': ['by', 'provided', 'available', 'published', 'recommended'], 'treatment': ['therapy', 'surgery', 'published', 'requires', 'by']}
TF-IDF   : [('d1', 0.0), ('d2', 0.0), ('d3', 0.0), ('d4', 0.0)] | R-precision = 0.00  (liên quan: ['d1', 'd10', 'd12', 'd2', 'd3'])
Embedding: [('d1', 0.67), ('d3', 0.66), ('d2', 0.54), ('d12', 0.48)] | R-precision = 1.00

=== Query: 'automobile engine' ===
từ liên quan (embedding): {'automobile': ['car', 'bus', 'truck', 'light', 'engine'], 'engine': ['fuel', 'update', 'needs', 'family', 'road']}
TF-IDF   : [('d5', 0.46), ('d1', 0.0), ('d2', 0.0), ('d3', 0.0)] | R-precision = 1.00  (liên quan: ['d5'])
Embedding: [('d5', 0.89), ('d1', 0.49), ('d8', 0.48), ('d6', 0.45)] | R-precision = 1.00

=== Query: 'paris capital' ===
từ liên quan (embedding): {'paris': ['hanoi', 'berlin', 'rome', 'madrid', 'tokyo'], 'capital': ['travelled', 'city', 'proud', 'are', 'year']}
TF-IDF   : [('d11', 0.82), 

**Nhận xét.** TF-IDF chỉ khớp được khi document chứa *đúng từ* trong query; embedding bắc cầu qua các từ cùng ngữ cảnh

In [ ]:
GROUPS = {"medical": ["doctor","physician","nurse","surgeon","hospital","patient","disease","clinic","therapy","surgery","infection","cancer"],
          "animal": ["cat","dog","fish","milk","meat","bone","pet","mouse"],
          "vehicle": ["car","automobile","truck","bus","road","engine","driver","garage"],
          "tech": ["computer","laptop","software","keyboard","server","programmer","code","program"],
          "sport": ["football","player","goal","match","team","stadium","coach","fans"],
          "fruit": ["banana","apple","orange","mango","fruit","sweet","ripe","breakfast"],
          "royal": ["king","queen","prince","princess","palace","throne","crown","kingdom"],
          "geo": ["paris","france","rome","italy","berlin","germany","madrid","spain","hanoi","vietnam","tokyo","japan","capital","country"]}
words = [w for ws in GROUPS.values() for w in ws if w in wv]; truth = [g for g, ws in GROUPS.items() for w in ws if w in wv]
km = KMeans(n_clusters=len(GROUPS), n_init=10, random_state=0).fit(np.vstack([wv[w] for w in words]))
ari = adjusted_rand_score(truth, km.labels_); print(f"Adjusted Rand Index (KMeans k=8 vs nhóm chủ đề tự gán) = {ari:.3f}")
log("app_clustering", "kmeans k=8", f"{len(words)} words", "adjusted_rand_index", ari)
for c in range(len(GROUPS)): print(c, [w for w, l in zip(words, km.labels_) if l == c])

Adjusted Rand Index (KMeans k=8 vs nhóm chủ đề tự gán) = 0.860
0 ['computer', 'laptop', 'software', 'keyboard', 'server', 'programmer', 'code', 'program', 'football', 'player', 'goal']
1 ['fans', 'banana', 'apple', 'orange', 'mango', 'fruit', 'sweet', 'ripe', 'breakfast']
2 ['doctor', 'physician', 'nurse', 'surgeon', 'hospital', 'patient', 'disease', 'therapy', 'surgery', 'infection', 'cancer']
3 ['paris', 'france', 'rome', 'italy', 'berlin', 'germany', 'madrid', 'spain', 'hanoi', 'vietnam', 'tokyo', 'japan', 'capital', 'country']
4 ['king', 'queen', 'prince', 'princess', 'palace', 'throne', 'crown']
5 ['clinic', 'car', 'automobile', 'truck', 'bus', 'road', 'engine', 'driver', 'garage']
6 ['match', 'team', 'stadium', 'coach', 'kingdom']
7 ['cat', 'dog', 'fish', 'milk', 'meat', 'bone', 'pet', 'mouse']


## Prediction

In [14]:
def small_corpus_trial(n_sent, epochs, trial):
    r = random.Random(1000 + trial); sub = [tokens[i] for i in r.sample(range(len(tokens)), n_sent)]
    m, _ = train(sub, epochs=epochs, seed=trial, min_count=2); kv = m.wv
    if "doctor" not in kv or "physician" not in kv: return None
    others = [kv.similarity("doctor", w) for w in kv.index_to_key if w != "doctor"]
    rank = [w for w, _ in kv.most_similar("doctor", topn=len(kv) - 1)].index("physician") + 1
    return float(kv.similarity("doctor", "physician")), float(np.mean(others)), rank / (len(kv) - 1), len(kv)
rows = []
for ep in (10, 100):
    res = [small_corpus_trial(100, ep, t) for t in range(30)]; ok = [x for x in res if x]
    a = np.array(ok)
    rows.append({"epochs": ep, "n_sent": 100, "cả 2 từ trong vocab": f"{len(ok)}/30", "vocab (mean)": round(a[:, 3].mean()),
                 "cos(doctor,physician) mean": round(a[:, 0].mean(), 3), "std": round(a[:, 0].std(), 3),
                 "min": round(a[:, 0].min(), 3), "max": round(a[:, 0].max(), 3),
                 "cos(doctor, mọi từ khác) mean": round(a[:, 1].mean(), 3), "rank chuẩn hoá của physician (0=gần nhất)": round(a[:, 2].mean(), 3)})
    s_ = f"100 sentences, epochs={ep}"
    log("pred4_small_corpus", s_, "doctor-physician", "cosine_mean_30trials", a[:, 0].mean()); log("pred4_small_corpus", s_, "doctor-physician", "cosine_std_30trials", a[:, 0].std())
    log("pred4_small_corpus", s_, "doctor-all_other_words", "cosine_mean_30trials", a[:, 1].mean())
    log("pred4_small_corpus", s_, "physician", "normalized_rank_mean", a[:, 2].mean()); log("pred4_small_corpus", s_, "doctor-physician", "both_in_vocab_rate", len(ok) / 30)
full_others = np.mean([wv.similarity("doctor", w) for w in wv.index_to_key if w != "doctor"])
allr = [w for w, _ in wv.most_similar("doctor", topn=len(wv) - 1)].index("physician") + 1
rows.append({"epochs": 10, "n_sent": 30000, "cả 2 từ trong vocab": "1/1", "vocab (mean)": len(wv), "cos(doctor,physician) mean": round(float(wv.similarity("doctor","physician")), 3),
             "std": np.nan, "min": np.nan, "max": np.nan, "cos(doctor, mọi từ khác) mean": round(float(full_others), 3), "rank chuẩn hoá của physician (0=gần nhất)": round(allr / (len(wv) - 1), 3)})
display(pd.DataFrame(rows).T)

,0,1,2
epochs,10,100,10
n_sent,100,100,30000
cả 2 từ trong vocab,28/30,28/30,1/1
vocab (mean),120,120,212
"cos(doctor,physician) mean",0.249,0.981,0.737
std,0.108,0.014,NaN
min,0.057,0.928,NaN
max,0.501,0.996,NaN
"cos(doctor, mọi từ khác) mean",0.207,0.955,0.228
rank chuẩn hoá của physician (0=gần nhất),0.392,0.169,0.014


## 13. Error analysis

In [ ]:
cands = ["physician","nurse","surgeon","hospital","patient","disease","clinic","computer","banana","car"]
allrank = [w for w, _ in wv.most_similar("doctor", topn=len(wv) - 1)]
rows = []
for n in cands:
    both, shared = evidence("doctor", n, k=4)
    rows.append({"doctor→": n, "cosine": round(float(wv.similarity("doctor", n)), 3), "rank": allrank.index(n) + 1,
                 "#câu chứa cả hai": both, "freq(neighbor)": cnt[n], "context chung": ", ".join(shared)})
    log("error_analysis", "sg d=100 w=5", f"doctor->{n}", "cosine", wv.similarity("doctor", n))
    log("error_analysis", "sg d=100 w=5", f"doctor->{n}", "rank_among_%d" % (len(wv) - 1), allrank.index(n) + 1)
display(pd.DataFrame(rows))
print("Top-10 doctor:", [(w, round(s, 2)) for w, s in wv.most_similar("doctor", topn=10)])
def distinct_ctx(a, b, k=6):
    ra, rb = X_ns[V_ns[a]], X_ns[V_ns[b]]
    return [f"{inv_ns[j]}({int(ra[j])})" for j in np.argsort(-ra) if rb[j] == 0 and ra[j] > 0][:k]
for other in ("nurse", "physician"):
    print(f"context xuất hiện với doctor nhưng KHÔNG với {other}:", distinct_ctx("doctor", other))
    print(f"context xuất hiện với {other} nhưng KHÔNG với doctor:", distinct_ctx(other, "doctor"))
print("rank của 'doctor' trong danh sách láng giềng của 'hospital':", [w for w, _ in wv.most_similar("hospital", topn=len(wv) - 1)].index("doctor") + 1, "/", len(wv) - 1)
print("Số câu có 'doctor':", cnt["doctor"], "| 'physician':", cnt["physician"], "| 'nurse':", cnt["nurse"], "| 'hospital':", cnt["hospital"])

,doctor→,cosine,rank,#câu chứa cả hai,freq(neighbor),context chung
0,physician,0.737,3,0,1725,"patient(503), recommended(161), provided(161),..."
1,nurse,0.989,1,0,1366,"patient(499), symptoms(142), explained(142), w..."
2,surgeon,0.791,2,1,876,"patient(299), surgery(111), about(111), asked(..."
3,hospital,0.389,22,460,2558,"patient(624), new(162), treated(158), hired(158)"
4,patient,0.439,11,766,3209,"examined(172), recommended(169), treated(158),..."
5,disease,0.410,17,120,723,"hospital(135), prescribed(115), patient(110), ..."
6,clinic,0.326,37,0,552,"disease(120), infection(116), diabetes(112), s..."
7,computer,0.218,90,1,447,"new(104), wrote(2), programmer(2), update(1)"
8,banana,0.237,77,1,507,"new(4), visited(1), update(1), sweet(1)"
9,car,0.253,64,0,742,"new(162), patient(2), asked(2), update(1)"


Top-10 doctor: [('nurse', 0.99), ('surgeon', 0.79), ('physician', 0.74), ('rest', 0.57), ('should', 0.55), ('home', 0.52), ('shifts', 0.46), ('cancer', 0.45), ('visited', 0.45), ('infection', 0.44)]
context xuất hiện với doctor nhưng KHÔNG với nurse: ['programmer(2)', 'doctor(2)', 'wrote(2)', 'paris(1)', 'loved(1)', 'people(1)']
context xuất hiện với nurse nhưng KHÔNG với doctor: ['mango(1)', 'eats(1)', 'fish(1)', 'left(1)', 'dog(1)', 'bus(1)']
context xuất hiện với doctor nhưng KHÔNG với physician: ['said(183)', 'should(177)', 'rest(177)', 'visited(154)', 'long(150)', 'home(150)']
context xuất hiện với physician nhưng KHÔNG với doctor: ['clinical(356)', 'published(198)', 'study(194)', 'reviewed(164)', 'evidence(162)', 'tired(2)']
rank của 'doctor' trong danh sách láng giềng của 'hospital': 19 / 211
Số câu có 'doctor': 1769 | 'physician': 1725 | 'nurse': 1366 | 'hospital': 2558


## 14. Polysemy — `bank`

In [ ]:
print("Top-10 láng giềng của 'bank':", [(w, round(s, 2)) for w, s in wv.most_similar("bank", topn=10)])
money = wv.get_mean_vector(["money","loan","interest","account","deposited","approved"]); river = wv.get_mean_vector(["river","water","boat","flooded","swam","sat"])
print(f"\nVector tĩnh 'bank':  cos với prototype-tiền = {cosine_similarity(wv['bank'], money):.3f} | cos với prototype-sông = {cosine_similarity(wv['bank'], river):.3f}")
log("polysemy_bank", "static", "bank", "cos_money_prototype", cosine_similarity(wv['bank'], money)); log("polysemy_bank", "static", "bank", "cos_river_prototype", cosine_similarity(wv['bank'], river))
for s in ["i deposited money in the bank", "we sat on the river bank", "the bank approved the loan", "fish swam near the river bank"]:
    ctx = wv.get_mean_vector([w for w in s.split() if w not in STOP and w != "bank" and w in wv])
    cm, cr = cosine_similarity(ctx, money), cosine_similarity(ctx, river)
    print(f"{s!r:40s} ngữ cảnh→tiền={cm:+.2f}  ngữ cảnh→sông={cr:+.2f}  ⇒ {'tiền' if cm > cr else 'sông'}")
    log("polysemy_bank", s, "context_mean", "cos_money_minus_river", cm - cr)

Top-10 láng giềng của 'bank': [('money', 0.56), ('we', 0.56), ('save', 0.56), ('an', 0.55), ('river', 0.54), ('opened', 0.54), ('account', 0.54), ('deposited', 0.53), ('interest', 0.52), ('loan', 0.52)]

Vector tĩnh 'bank':  cos với prototype-tiền = 0.627 | cos với prototype-sông = 0.575
'i deposited money in the bank'          ngữ cảnh→tiền=+0.88  ngữ cảnh→sông=+0.55  ⇒ tiền
'we sat on the river bank'               ngữ cảnh→tiền=+0.63  ngữ cảnh→sông=+0.91  ⇒ sông
'the bank approved the loan'             ngữ cảnh→tiền=+0.91  ngữ cảnh→sông=+0.66  ⇒ tiền
'fish swam near the river bank'          ngữ cảnh→tiền=+0.63  ngữ cảnh→sông=+0.87  ⇒ sông


## 15. Reflection — từ Word2Vec đến Transformer

| Representation | Context-dependent? | Sparse/Dense | Một từ có nhiều vector? |
|---|---|---|---|
| TF-IDF | Không (trọng số theo thống kê toàn corpus; vector là của *văn bản*, không phải của từ) | Sparse | Không |
| Co-occurrence | Không (context chỉ để *tạo* vector, vector tạo xong là cố định) | Sparse (có thể nén bằng SVD thành dense) | Không |
| Word2Vec | Không (static) | Dense | Không – mỗi từ đúng một vector |
| Contextual embedding | **Có** | Dense | **Có** – vector khác nhau theo từng câu |
